## Train-Test Split

In [1]:
import pandas as pd

df = pd.read_csv('../final_data/netflix_final.csv')

(df['User_ID'].value_counts() == 1).sum()

np.int64(86373)

We can see that in our current filteredd df we got about 87k users with only 1 rating, hence it could possibly throw a value-error if we try the train_test_split on it
so I basically remove the single rated users, might sound alot but in reality, its just 87k reviews
hence with this basic math, (2 mil - 87 k) is not that big deal.

In [2]:
from sklearn.model_selection import train_test_split

# cleanuppp
valid_users = df['User_ID'].value_counts()
valid_users = valid_users[valid_users >= 2].index
df = df[df['User_ID'].isin(valid_users)].reset_index(drop=True)

# Save this into the exisitng netflix_final.csv
# split 80/20 stratified by user so every user appears in both sets
train_data, test_data = train_test_split(
    df,
    test_size=0.20,
    stratify=df['User_ID'],
    random_state=42
)

print(f"Training Set Size: {len(train_data):,} rows")
print(f"Testing Set Size : {len(test_data):,} rows")

print(f"Unique users in Train: {train_data['User_ID'].nunique():,}")
print(f"Unique users in Test : {test_data['User_ID'].nunique():,}")

train_data.to_csv("../final_data/Train_Data.csv", index=False)
test_data.to_csv("../final_data/Test_Data.csv", index=False)

Training Set Size: 1,506,358 rows
Testing Set Size : 376,590 rows
Unique users in Train: 240,460
Unique users in Test : 184,604


In [5]:
# making sure, all the ratings are from 1 till 5...
print(df['Rating'].min(), df['Rating'].max())

1 5


I will be using the surprise library, source : https://surpriselib.com/

*"Provide various ready-to-use prediction algorithms such as baseline algorithms, neighborhood methods, matrix factorization-based ( SVD, PMF, SVD++, NMF), and many others. Also, various similarity measures (cosine, MSD, pearson…) are built-in."*

Hence this is very useful for us and also mantains the industry grade standard

In [ ]:
from surprise import Dataset, Reader

# Formatting the data
# Defining the rating scale (1 to 5)
reader = Reader(rating_scale=(1, 5))

# we got 3 columns User, Movie, Rating...
train_subset = train_data[['User_ID', 'Movie_ID', 'Rating']]
test_subset  = test_data[['User_ID', 'Movie_ID', 'Rating']]

# Convert the dataframe into a Surprise Dataset
# and then I build the final train set (it converts into a sparse matrix)
# meaning that instead of having several huge number of missing vals in these ratings...
# since some users might not rate some random rating, so yea this problem is solved using this..
train_dataset = Dataset.load_from_df(train_subset, reader)
trainset_surprise = train_dataset.build_full_trainset()

# THE FINAL TESTTT_SET
# Surprise expects the test data as a list of tuples such as [(User, Movie, Rating), ...]
testset_surprise = [tuple(x) for x in test_subset.to_numpy()]

print(f"Surprise Training Matrix: {trainset_surprise.n_users:,} users x {trainset_surprise.n_items:,} movies.")
print(f"Surprise Testing: {len(testset_surprise):,} for prediction.")

Surprise Training Matrix: 240,460 users x 4,711 movies.
Surprise Testing: 376,590 for prediction.


#### Here for this project I have choosen to use the **Singular Value Decomposition (SVD)** & **Item-Based Collaborative Filtering** as my recommendation models


In [7]:
from surprise import SVD, KNNWithMeans
import time # for numbers related to the computational efficiency...

# MODEL 1: SVD (Matrix Factorization) 
print("Model1: Training SVD...")
start_time = time.time()
svd_model = SVD(random_state=42) 
svd_model.fit(trainset_surprise)
print(f"SVD Training took: {time.time() - start_time:.2f} seconds.\n")

# MODEL 2: Item-Based Collaborative Filtering 
print("Model2: Training Item-Based CF (KNN)...")
start_time = time.time()
sim_options = {'name': 'cosine', 'user_based': False}
knn_model = KNNWithMeans(k=20, min_k=5, sim_options=sim_options, verbose=False)
knn_model.fit(trainset_surprise)
print(f"Item-Based CF Training took: {time.time() - start_time:.2f} seconds.\n")

print("Gen. predictions on the test set, using the currently built models, including the time taken,")
# Timing the SVD Predictions
start_time = time.time()
svd_predictions = svd_model.test(testset_surprise)
print(f"SVD predictions gen. in: {time.time() - start_time:.2f} seconds.")

# Timing the  KNN Predictions
start_time = time.time()
knn_predictions = knn_model.test(testset_surprise)
print(f"KNN predictions gen. in: {time.time() - start_time:.2f} seconds.")

Model1: Training SVD...
SVD Training took: 13.53 seconds.

Model2: Training Item-Based CF (KNN)...
Item-Based CF Training took: 2.83 seconds.

Gen. predictions on the test set, using the currently built models, including the time taken,
SVD predictions gen. in: 1.88 seconds.
KNN predictions gen. in: 6.94 seconds.
